# Kontrola środowiska

Sprawdza po kolei każdy komponent potoku: czy w ogóle się ładuje, na jakim urządzeniu liczy, ile zajmuje pamięci karty i jak długo trwa.

**Wymaga:** kernela wskazującego środowisko `wideo` (prawy górny róg). Jeśli go nie ma na liście, zajrzyj do `notebooks/README.md`.

**Zapisuje:** `results/measurements/<nazwa>_<data>_<godzina>.json` - po jednym pliku na pomiar. Nazwa niesie datę i godzinę, więc kolejne przebiegi zostają obok siebie zamiast się nadpisywać.

In [ ]:
# Pylance scopes '# pyright:' comments to a single cell, so cells calling
# libraries with incomplete type stubs carry their own pragma.
import sys
from pathlib import Path

# working directory should be the repository root; fall back to finding src/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.utils.notebook import (
    find_clip,
    measure,
    nvidia_smi,
    release,
    save_measurement,
    setup,
    summary,
    test_frame,
    vram,
)

environment = setup()

---
## A. Karta, sterownik, biblioteki

Trzy warunki muszą być spełnione naraz: wersja PyTorcha kończy się na `+cu130`, `torch.cuda.is_available()` zwraca prawdę, a lista architektur zawiera `sm_120`. Wariant `cu126` uruchomi się i policzy, tylko na procesorze - bez ostrzeżenia.

In [ ]:
print(nvidia_smi())

W kolumnie `memory.used` widać pamięć zajętą przez pulpit Windows, kompozytor okien i akcelerację przeglądarki. Pomiar z 4 sierpnia dał 4759 MiB przy bezczynnym systemie, co zostawiało około 11,3 GiB, za mało dla LLaVY-1.5-7B. Po przepięciu monitora na układ zintegrowany procesora `display_active` ma pokazywać `Disabled`, a `memory.used` kilkadziesiąt MiB.

**To jest ten pomiar, który trzeba powtórzyć po przepięciu.**

In [ ]:
state = vram("idle state")
save_measurement("idle_state", state)

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import importlib.metadata as pkg_metadata

# libraries listed in the thesis plus those whose exact build decides GPU
# correctness (torchvision, av, accelerate)
PACKAGES = [
    "torch", "torchvision", "transformers", "open_clip_torch", "ultralytics",
    "onnxruntime-gpu", "insightface", "hsemotion-onnx", "faiss-cpu",
    "pytorchvideo", "transnetv2-pytorch", "accelerate", "numpy", "pandas",
    "opencv-python", "av", "matplotlib", "pydantic",
]
versions = {}
for package in PACKAGES:
    try:
        versions[package] = pkg_metadata.version(package)
    except pkg_metadata.PackageNotFoundError:
        versions[package] = "MISSING"
    print(f"{package:<22}{versions[package]}")

# FFmpeg is a system tool, not a Python package
import subprocess

try:
    banner = subprocess.run(["ffprobe", "-version"], capture_output=True, check=False,
                            text=True, encoding="utf-8", errors="replace")
    versions["ffmpeg"] = banner.stdout.split()[2] if banner.returncode == 0 else "ERROR"
except FileNotFoundError:
    versions["ffmpeg"] = "MISSING"
print(f"{'ffmpeg':<22}{versions['ffmpeg']}")

save_measurement("library_versions", versions)

### Koszt precyzji pojedynczej

Rozdział 5 zakłada, że wszystkie modele poza LLaVĄ pracują w precyzji pojedynczej. Ta komórka mierzy, ile to kosztuje: mnożenie macierzy 4096x4096 w trzech precyzjach. Jeśli fp16 wychodzi wyraźnie szybsze od fp32, rdzenie tensorowe działają. Wynik rzędu pojedynczych TFLOPS w fp32 oznacza, że liczy procesor.

In [ ]:
import time

import torch


def throughput(dtype, n=4096, repeats=20):
    a = torch.randn(n, n, device="cuda", dtype=dtype)
    b = torch.randn(n, n, device="cuda", dtype=dtype)
    for _ in range(3):          # warm-up
        _ = a @ b
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        _ = a @ b
    torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) / repeats
    del a, b
    torch.cuda.empty_cache()
    return elapsed, 2 * n**3 / elapsed / 1e12


throughputs = {}
for name, dtype in [("fp32", torch.float32), ("fp16", torch.float16),
                    ("bf16", torch.bfloat16)]:
    elapsed, tflops = throughput(dtype)
    throughputs[name] = {"time_ms": round(elapsed * 1000, 2), "tflops": round(tflops, 1)}
    print(f"{name}: {elapsed * 1000:7.1f} ms   {tflops:6.1f} TFLOPS")

ratio = throughputs["fp16"]["tflops"] / throughputs["fp32"]["tflops"]
throughputs["fp16_to_fp32"] = round(ratio, 2)
print(f"\nfp16 faster than fp32 by {ratio:.1f}x")

save_measurement("matmul_throughput", throughputs)

---
## B. ONNX Runtime i twarze (RetinaFace + ArcFace)

Drugi, niezależny silnik wykonania - ma własny build CUDA i może zawieść osobno. Wydania `onnxruntime-gpu` starsze niż 1.27 nie mają kerneli `sm_120` i po cichu przenoszą obliczenia na procesor. Pakiet `onnxruntime` w wariancie procesorowym nie może współistnieć z `onnxruntime-gpu` - oba dostarczają ten sam moduł.

In [ ]:
import onnxruntime as ort

if hasattr(ort, "preload_dlls"):
    # ORT >= 1.21: loads the cublas/cudnn DLLs from the pip nvidia-* packages;
    # without it a session created before importing torch falls back to the CPU
    ort.preload_dlls()

print("version   :", ort.__version__)
print("providers :", ort.get_available_providers())

Potok używa dwóch docelowych modeli ONNX: detektora **RetinaFace-ResNet-50** (`data/models/retinaface_r50.onnx`, konwersja wg `docs/00_srodowisko.md`) oraz embeddingów tożsamości **ArcFace R100 `glintr100`** z pakietu `antelopev2` (`~/.insightface/models/antelopev2/`). Poniżej kontrola silnika: obie sieci ładują się w ONNX Runtime na CUDA i wykonują surowy przebieg. Dekodowanie wyjść detektora (ramki + punkty) powstanie w module `src/features/faces.py` przy eksperymencie E5 - tu sprawdzany jest wyłącznie silnik wykonania.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
from pathlib import Path

import numpy as np
import onnxruntime as ort

if hasattr(ort, "preload_dlls"):
    # ORT >= 1.21: loads the cublas/cudnn DLLs from the pip nvidia-* packages;
    # without it a session created before importing torch falls back to the CPU
    ort.preload_dlls()

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
RETINAFACE = ROOT / "data" / "models" / "retinaface_r50.onnx"
GLINTR100 = Path.home() / ".insightface" / "models" / "antelopev2" / "glintr100.onnx"
PROVIDERS = ["CUDAExecutionProvider", "CPUExecutionProvider"]

detector = arcface = None
if not RETINAFACE.exists():
    print(f"MISSING {RETINAFACE} - convert it per docs/00_srodowisko.md")
else:
    with measure("RetinaFace R50: loading") as p_retina:
        detector = ort.InferenceSession(str(RETINAFACE), providers=PROVIDERS)
    print(f"  provider  : {detector.get_providers()[0]}")
    print(f"  input     : {detector.get_inputs()[0].name} {detector.get_inputs()[0].shape}")

if not GLINTR100.exists():
    print(f"MISSING {GLINTR100} - download the package: "
          "python -c \"from insightface.app import FaceAnalysis; FaceAnalysis(name='antelopev2')\"")
else:
    with measure("ArcFace glintr100: loading") as p_arcface:
        arcface = ort.InferenceSession(str(GLINTR100), providers=PROVIDERS)
    print(f"  provider  : {arcface.get_providers()[0]}")
    print(f"  input     : {arcface.get_inputs()[0].name} {arcface.get_inputs()[0].shape}")

Surowy przebieg detektora i enkodera tożsamości.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
# raw forward passes - execution-engine check, no output decoding yet
results = {}

if detector is not None:
    shape = detector.get_inputs()[0].shape          # [1, 3, H, W]; H/W may be dynamic
    height = shape[2] if isinstance(shape[2], int) else 640
    width = shape[3] if isinstance(shape[3], int) else 640
    frame = np.array(test_frame().resize((width, height)))[:, :, ::-1]   # BGR
    means = np.array((104, 117, 123), dtype=np.float32)   # BGR means of the model
    blob = (frame.astype(np.float32) - means).transpose(2, 0, 1)[None]
    with measure("RetinaFace R50: frame") as p_det_frame:
        outputs = detector.run(None, {detector.get_inputs()[0].name: blob})
    print("  outputs:", [tuple(o.shape) for o in outputs])   # loc / conf / landms
    results["retinaface"] = p_det_frame

if arcface is not None:
    face = np.random.default_rng(0).integers(0, 255, (112, 112, 3)).astype(np.float32)
    blob = ((face - 127.5) / 127.5).transpose(2, 0, 1)[None]
    with measure("ArcFace glintr100: aligned crop") as p_arc_face:
        embedding = arcface.run(None, {arcface.get_inputs()[0].name: blob})[0]
    print("  identity embedding dim:", embedding.shape[-1])
    results["arcface"] = p_arc_face

if results:
    save_measurement("faces_onnx", results)
release("detector", "arcface", "outputs", "embedding")

---
## C. OpenCLIP ViT-H/14 - mechanizm bazowy

Konfiguracja bazowa całego badania. Sprawdzane są trzy rzeczy: model się ładuje, obraz i tekst lądują we wspólnej przestrzeni, a podobieństwo kosinusowe rozróżnia opisy trafne od nietrafnych. Wagi pobierają się przy pierwszym uruchomieniu.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import open_clip
import torch

with measure("OpenCLIP ViT-H/14: loading") as p_openclip:
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-H-14", pretrained="laion2b_s32b_b79k", device="cuda",
    )
    tokenizer = open_clip.get_tokenizer("ViT-H-14")
    model.eval()

vram("after loading OpenCLIP")

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
image = test_frame()
descriptions = [
    "a person walking down a street",
    "a dog running on grass",
    "an empty room at night",
]

with torch.no_grad(), measure("OpenCLIP: encoding image and text") as p_encoding:
    inputs = preprocess(image).unsqueeze(0).to("cuda")
    image_features = model.encode_image(inputs)
    text_features = model.encode_text(tokenizer(descriptions).to("cuda"))
    image_features /= image_features.norm(dim=-1, keepdim=True)
    text_features /= text_features.norm(dim=-1, keepdim=True)
    similarity = (image_features @ text_features.T).squeeze(0)

print(f"embedding dim: {image_features.shape[-1]}\n")
for desc, value in sorted(zip(descriptions, similarity.tolist()), key=lambda x: -x[1]):
    print(f"{value:6.3f}   {desc}")

save_measurement("openclip", {**p_encoding, "dim": int(image_features.shape[-1])})

In [ ]:
release("model", "preprocess", "tokenizer", "image_features", "text_features")

---
## D. TransNetV2 - granice ujęć

Komponent segmentacji dla eksperymentu E1 (podział na ujęcia kontra stałe okna). Wagi są zaszyte w pakiecie, więc nic się nie pobiera. Wejściem jest sekwencja klatek przeskalowanych do 48x27 - taki rozmiar wymusza architektura sieci.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import torch
from transnetv2_pytorch import TransNetV2

with measure("TransNetV2: loading") as p_transnet:
    transnet = TransNetV2().eval().to("cuda")

with torch.no_grad(), measure("TransNetV2: 100 frames") as p_boundaries:
    inputs = torch.zeros(1, 100, 27, 48, 3, dtype=torch.uint8, device="cuda")
    single, all_frames = transnet(inputs)

print("single-frame output:", getattr(single, "shape", type(single)))
print("all-frames output  :", type(all_frames))

save_measurement("transnetv2", p_boundaries)
release("transnet", "single", "all_frames", "inputs")

---
## E. LLaVA-1.5-7B w precyzji połowicznej - sprawdzenie graniczne

Najważniejsze sprawdzenie w tym notatniku i jedyne, które może nie przejść. Model potrzebuje około 15 GB, karta ma 16 GB, więc wynik zależy od tego, ile pamięci zabiera pulpit (sekcja A). Jeśli tu wyskoczy `OutOfMemoryError`, plan awaryjny w kolejności: `device_map="auto"` z przeniesieniem ostatnich warstw do pamięci operacyjnej, a dopiero na końcu kwantyzacja - bo ta zmienia wagi, więc wpływa na mierzoną wielkość i musiałaby zostać odnotowana w metodologii.

Pobranie wag przy pierwszym uruchomieniu to około 14 GB.

**Uwaga na wersję biblioteki.** W środowisku jest `transformers` 5.x, gdzie treść wiadomości jest listą słowników, a nie łańcuchem znaków. Przykłady z sieci pisane pod wersję 4 nie zadziałają.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import torch
from transformers import AutoProcessor, LlavaForConditionalGeneration

MODEL = "llava-hf/llava-1.5-7b-hf"
# Eventually add revision="<commit hash>" -- the model name alone is not enough
# to reproduce the result, because Hugging Face repositories get updated.

with measure("LLaVA: loading") as p_llava:
    processor = AutoProcessor.from_pretrained(MODEL)
    llava = LlavaForConditionalGeneration.from_pretrained(
        MODEL,
        dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )

vram("after loading LLaVA")
print("layer placement:", set(llava.hf_device_map.values())
      if hasattr(llava, "hf_device_map") else "everything on one device")

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
image = test_frame()

conversation = [{
    "role": "user",
    "content": [
        {"type": "image", "image": image},
        {"type": "text", "text": "Describe in one sentence what is happening in this image."},
    ],
}]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(llava.device, torch.float16)

with measure("LLaVA: generating a description") as p_desc:
    outputs = llava.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=64)

prompt_length = inputs["input_ids"].shape[-1]
desc = processor.decode(outputs[0][prompt_length:], skip_special_tokens=True)
print(desc.strip())

Rozdział 5 deklaruje, że model wybiera zawsze najbardziej prawdopodobne słowo, dzięki czemu ta sama klatka daje przy każdym uruchomieniu identyczny opis. Poniżej dwa przebiegi porównane znak po znaku.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
repeats = []
for _ in range(2):
    outputs = llava.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=64)
    repeats.append(
        processor.decode(outputs[0][prompt_length:], skip_special_tokens=True).strip()
    )

identical = repeats[0] == repeats[1]
print("descriptions identical:", identical)
if not identical:
    print("\n1:", repeats[0])
    print("2:", repeats[1])

save_measurement("llava_fp16", {
    **p_desc,
    "load_time": p_llava["time"],
    "load_peak_gib": p_llava.get("peak_allocated_gib"),
    "deterministic": identical,
    "desc": repeats[0],
})

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
release("llava", "processor", "inputs", "outputs")

Podgląd klatki, na której sprawdzana jest LLaVA-1.5-7B.

In [ ]:
print(find_clip())
test_frame()

test_frame().save("test_frame.png")

---
## F. SlowFast R50 - modelowanie ruchu

Cechy ruchu dla eksperymentu E3. Świadomie przez `pytorchvideo`, a nie PySlowFast: to drugie ciągnie za sobą Detectron2, którego kompilacja na Windows jest udręką. Wejściem są dwie ścieżki o różnej gęstości próbkowania w czasie - wolna 8 klatek, szybka 32.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import torch
from pytorchvideo.models.hub import slowfast_r50

with measure("SlowFast R50: loading") as p_slowfast:
    slowfast = slowfast_r50(pretrained=True).eval().to("cuda")

with torch.no_grad(), measure("SlowFast R50: forward pass") as p_motion:
    slow_pathway = torch.randn(1, 3, 8, 224, 224, device="cuda")
    fast_pathway = torch.randn(1, 3, 32, 224, 224, device="cuda")
    result = slowfast([slow_pathway, fast_pathway])

print("output shape:", tuple(result.shape))

save_measurement("slowfast_r50", p_motion)
release("slowfast", "result", "slow_pathway", "fast_pathway")

---
## G. YOLOE - detekcja obiektów

Wariant z przyrostkiem `-pf` ma słownik 4585 klas zaszyty w wagach i nie wymaga pakietu `clip` ani enkodera tekstu. Komórka niżej sprawdza właśnie ten tryb, bo nic nie doinstalowuje.

Tryb z promptem tekstowym (`set_classes`) przy pierwszym wywołaniu ściąga pakiet `clip` z forka Ultralytics oraz enkoder `mobileclip_blt.ts` o rozmiarze 572 MB - dlatego jest w osobnej, opcjonalnej komórce. Nie instalować pakietu `clip` z PyPI: to zupełnie inny projekt, a Ultralytics sprawdza wyłącznie, czy moduł o tej nazwie istnieje.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false, reportIndexIssue=false
import numpy as np
from ultralytics import YOLOE

with measure("YOLOE prompt-free: loading") as p_yoloe:
    yoloe = YOLOE("yoloe-11l-seg-pf.pt")

frame = np.array(test_frame())

with measure("YOLOE prompt-free: frame") as p_object_detection:
    results = yoloe.predict(frame, verbose=False)

result = results[0]
boxes = result.boxes if result.boxes is not None else []
print(f"detected objects: {len(boxes)}")
print(f"vocabulary size : {len(result.names)} classes")
for box in boxes[:5]:
    print(f"  {result.names[int(box.cls)]:<20} {float(box.conf):.2f}")

save_measurement("yoloe_prompt_free", {
    **p_object_detection,
    "object_count": len(boxes),
    "vocab_size": len(result.names),
})
release("yoloe", "results", "result", "boxes")

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false, reportIndexIssue=false
# Text-prompt mode. The first run pulls the mobileclip_blt.ts text encoder (572 MB)
# and needs `pip install git+https://github.com/ultralytics/CLIP.git` beforehand, so
# it stays off unless switched on here and a Run All never triggers the download.
RUN_PROMPTED = False

if RUN_PROMPTED:
    from ultralytics import YOLOE

    yoloe = YOLOE("yoloe-11l-seg.pt")           # without -pf: the prompted weights
    yoloe.set_classes(["person", "traffic light", "coffee mug"])
    results = yoloe.predict(frame, verbose=False)
    print(len(results[0].boxes), "objects for the given phrases")
else:
    print("prompted mode off - set RUN_PROMPTED = True to run it")

---
## H. FAISS na procesorze - indeks wektorowy

Wariant procesorowy wystarcza dla skali tej pracy. Poniżej indeks 50 tysięcy wektorów o wymiarze embeddingu OpenCLIP ViT-H/14 i pomiar czasu pojedynczego zapytania - liczba potrzebna do rozdziału o kosztach, i argument w uzasadnieniu, dlaczego wariant na karcie nie jest tu potrzebny.

In [ ]:
import faiss
import numpy as np

COUNT, DIM, NEIGHBORS = 50_000, 1024, 10

rng = np.random.default_rng(0)
base = rng.standard_normal((COUNT, DIM), dtype=np.float32)
faiss.normalize_L2(base)

index = faiss.IndexFlatIP(DIM)
index.add(base)

queries = base[:100].copy()
start = time.perf_counter()
distances, positions = index.search(queries, NEIGHBORS)
query_time = (time.perf_counter() - start) / len(queries) * 1000

print(f"FAISS threads     : {faiss.omp_get_max_threads()}")
print(f"vectors in index  : {index.ntotal}")
print(f"query time        : {query_time:.3f} ms")
print(f"sanity check      : {'OK' if (positions[:, 0] == np.arange(100)).all() else 'FAIL'}"
      "  (the nearest neighbour of a base vector is itself)")

save_measurement("faiss_cpu", {
    "vector_count": COUNT,
    "dim": DIM,
    "neighbors": NEIGHBORS,
    "query_time_ms": round(query_time, 3),
    "threads": faiss.omp_get_max_threads(),
})

---
## I. Dekodowanie nagrania - wąskie gardło potoku

Podczas przetwarzania procesor będzie obciążony, a karta częściowo bezczynna. To zachowanie prawidłowe: dekodowanie wykonuje FFmpeg na procesorze. Pomiar poniżej pokazuje, ile klatek na sekundę da się przeczytać z dysku - i uzasadnia decyzję z rozdziału 5, żeby dekodować raz i zapisywać klatki do pamięci podręcznej zamiast powtarzać dekodowanie dla każdego komponentu.

In [ ]:
import time

import cv2

clip = find_clip([ROOT / "data" / "processed" / "office"])
if clip is None:
    print("no videos in data/ -- skip this section")
else:
    reader = cv2.VideoCapture(str(clip))
    frame_count = int(reader.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = reader.get(cv2.CAP_PROP_FPS)
    width = int(reader.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(reader.get(cv2.CAP_PROP_FRAME_HEIGHT))

    start = time.perf_counter()
    read = 0
    while True:
        ok, _ = reader.read()
        if not ok:
            break
        read += 1
    elapsed = time.perf_counter() - start
    reader.release()

    print(f"file     : {clip.name}")
    print(f"video    : {width}x{height}, {fps:.1f} fps, {frame_count} frames")
    print(f"decoding : {elapsed:.2f} s  ({read / elapsed:.0f} frames/s, "
          f"{read / elapsed / max(fps, 1):.1f}x real time)")

    save_measurement("video_decoding", {
        "file": clip.name,
        "resolution": f"{width}x{height}",
        "frames": read,
        "time": round(elapsed, 2),
        "frames_per_second": round(read / elapsed, 1),
    })

---
## J. Zapotrzebowanie pamięci na wagi

Dokładna suma bajtów parametrów każdego modelu z rozdziału 5 - same wagi, bez narzutu. Precyzja zgodna z deklaracją pracy: wszystko w pojedynczej, LLaVA w połowicznej.

In [ ]:
# pyright: reportCallIssue=false, reportAttributeAccessIssue=false
import gc
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())

GB = 1000 ** 3          # decimal gigabytes


def torch_numel(net):
    return sum(p.numel() for p in net.parameters())


def meta_transformers(cls_name, checkpoint):
    """Parameter count from the architecture alone (meta device, no weights)."""
    import transformers
    from accelerate import init_empty_weights

    config = transformers.AutoConfig.from_pretrained(checkpoint)
    with init_empty_weights():
        net = getattr(transformers, cls_name)(config)
    count = torch_numel(net)
    del net
    return count


def openclip_numel(name):
    import open_clip

    net = open_clip.create_model(name, pretrained=None)   # random init, no download
    count = torch_numel(net)
    del net
    gc.collect()
    return count


def transnet_numel():
    from transnetv2_pytorch import TransNetV2
    return torch_numel(TransNetV2())


def slowfast_numel():
    from pytorchvideo.models.hub import slowfast_r50
    return torch_numel(slowfast_r50(pretrained=False))


def yolo_numel(weights):
    from ultralytics import YOLO

    net = YOLO(weights).model
    assert net is not None   # present after loading a .pt checkpoint
    return sum(p.numel() for p in net.parameters())


def torchscript_numel(path):
    """Parameter count of a serialised TorchScript archive.

    The file also carries the serialised graph, so its size on disk is not the
    weight budget; only the parameters are counted, as in every other row.
    """
    import torch

    net = torch.jit.load(str(path), map_location="cpu")
    count = sum(p.numel() for p in net.parameters())
    del net
    gc.collect()
    return count


def onnx_gb(path):
    """Bytes of the initializers (the weights) of an ONNX file."""
    import onnx
    from onnx import numpy_helper

    graph = onnx.load(str(path)).graph
    return sum(numpy_helper.to_array(t).nbytes for t in graph.initializer) / GB


ANTELOPE = Path.home() / ".insightface" / "models" / "antelopev2"
ROOT_MODELS = ROOT / "data" / "models"


def hsemotion_path(model_name="enet_b0_8_best_vgaf"):
    """Model path; the package downloads to ~/.hsemotion on first use."""
    from hsemotion_onnx.facial_emotions import get_model_path

    return Path(get_model_path(model_name))


ROWS = [
    ("TransNetV2", "--", lambda: 4 * transnet_numel() / GB),
    ("OpenCLIP", "ViT-H/14", lambda: 4 * openclip_numel("ViT-H-14") / GB),
    ("OpenCLIP", "ViT-B/32, openai weights", lambda: 4 * openclip_numel("ViT-B-32") / GB),
    ("BLIP", "large", lambda: 4 * meta_transformers(
        "BlipForConditionalGeneration", "Salesforce/blip-image-captioning-large") / GB),
    ("LLaVA-1.5", "7B, FP16", lambda: 2 * meta_transformers(
        "LlavaForConditionalGeneration", "llava-hf/llava-1.5-7b-hf") / GB),
    ("SlowFast", "R50", lambda: 4 * slowfast_numel() / GB),
    ("X-CLIP", "base/32, zero-shot", lambda: 4 * meta_transformers(
        "XCLIPModel", "microsoft/xclip-base-patch32") / GB),
    ("HSEmotion", "EfficientNet-B0", lambda: onnx_gb(hsemotion_path())),
    ("YOLO11", "large", lambda: 4 * yolo_numel("yolo11l.pt") / GB),
    ("YOLOE-11", "large", lambda: 4 * yolo_numel("yoloe-11l-seg-pf.pt") / GB),
    ("YOLOE-11", "large, promptable", lambda: 4 * yolo_numel("yoloe-11l-seg.pt") / GB),
    ("MobileCLIP-BLT", "YOLOE text encoder",
     lambda: 4 * torchscript_numel(ROOT / "mobileclip_blt.ts") / GB),
    ("RetinaFace", "ResNet-50", lambda: onnx_gb(ROOT_MODELS / "retinaface_r50.onnx")),
    ("ArcFace", "R100 (glintr100)", lambda: onnx_gb(ANTELOPE / "glintr100.onnx")),
]

rows = []
for model_name, variant, compute in ROWS:
    try:
        value = compute()
    except Exception as error:  # noqa: BLE001 - one failing row must not kill the table
        print(f"{model_name:<28} {variant:<24} ERROR: {type(error).__name__}: {error}")
        continue
    rows.append({"model": model_name, "variant": variant, "weights_gb": round(value, 3)})
    print(f"{model_name:<28} {variant:<24} {value:6.2f} GB")

save_measurement("vram_weights", {"rows": rows})

---
## Zestawienie

Wszystko, co zapisało się do `results/measurements/` - z tego notatnika i z poprzednich przebiegów.

In [ ]:
summary()